# DP-SGD: Differentially Private Stochastic Gradient Descent

DP-SGD (Abadi et al. 2016) is the standard algorithm for training neural networks with differential privacy. It adds two operations to the standard SGD loop: gradient clipping (bounding sensitivity) and noise injection (providing DP guarantee).

## The Algorithm

For each training step:
1. Sample a minibatch B (Poisson subsampling, not uniform)
2. Compute per-sample gradients (not the average — each sample separately)
3. Clip each per-sample gradient to L2 norm ≤ C (sensitivity bound)
4. Average the clipped gradients
5. Add Gaussian noise N(0, σ²C²/|B|) to the average
6. Update model parameters

The clipping step bounds how much any single training example can influence the model. The noise step provides the DP guarantee given that bound.

In [ ]:
import math, random
from dataclasses import dataclass

@dataclass
class DPSGDConfig:
    max_grad_norm: float = 1.0    # C: clipping threshold
    noise_multiplier: float = 1.1  # sigma: noise / clipping_norm
    batch_size: int = 256
    epochs: int = 10
    learning_rate: float = 0.01
    delta: float = 1e-5

def clip_gradient(grad: list, max_norm: float) -> list:
    norm = math.sqrt(sum(g*g for g in grad))
    if norm > max_norm:
        scale = max_norm / norm
        return [g * scale for g in grad]
    return grad

def add_gaussian_noise(grad: list, noise_std: float, rng) -> list:
    return [g + rng.gauss(0, noise_std) for g in grad]

def dp_sgd_step(per_sample_grads: list, config: DPSGDConfig, rng) -> list:
    clipped = [clip_gradient(g, config.max_grad_norm) for g in per_sample_grads]
    avg = [sum(g[i] for g in clipped) / len(clipped) for i in range(len(clipped[0]))]
    noise_std = config.noise_multiplier * config.max_grad_norm / len(per_sample_grads)
    noisy = add_gaussian_noise(avg, noise_std, rng)
    return noisy

def compute_epsilon(config: DPSGDConfig, n: int) -> float:
    # Simplified privacy accounting (Moments Accountant approximation)
    q = config.batch_size / n  # sampling ratio
    steps = (n // config.batch_size) * config.epochs
    # RDP-based approximation
    sigma = config.noise_multiplier
    # alpha = 2 (moment order)
    rdp_per_step = q**2 / (2 * sigma**2)
    total_rdp = rdp_per_step * steps
    # Convert RDP to (epsilon, delta)-DP
    epsilon = total_rdp + math.log(1/config.delta) / 1  # simplified
    return round(epsilon, 3)

rng = random.Random(42)
config = DPSGDConfig(max_grad_norm=1.0, noise_multiplier=1.1, batch_size=256, epochs=10)

# Simulate DP-SGD on a toy 2D problem
n_params = 2
weights = [0.0, 0.0]
n_train = 10000

def generate_batch(n, rng):
    return [[rng.gauss(0, 0.5) for _ in range(n_params)] for _ in range(n)]

print('DP-SGD Training Simulation:')
print(f'Config: max_grad_norm={config.max_grad_norm}, noise_multiplier={config.noise_multiplier}')
print(f'Privacy guarantee: ε={compute_epsilon(config, n_train)}, δ={config.delta}')

losses = []
for epoch in range(config.epochs):
    n_batches = n_train // config.batch_size
    epoch_loss = 0
    for _ in range(n_batches):
        batch_grads = generate_batch(config.batch_size, rng)
        update = dp_sgd_step(batch_grads, config, rng)
        weights = [w - config.learning_rate * u for w, u in zip(weights, update)]
        epoch_loss += sum(u*u for u in update)
    losses.append(epoch_loss / n_batches)

print(f'\nTraining loss over epochs:')
for i, loss in enumerate(losses):
    print(f'  Epoch {i+1}: {loss:.6f}')
print(f'Final weights: {[round(w,4) for w in weights]}')

## Privacy-Utility Tradeoff in DP-SGD

DP-SGD introduces a fundamental tension: stronger privacy (lower ε) requires more noise, which reduces model utility. The key levers:

**Noise multiplier σ**: higher σ = stronger privacy = more accuracy loss. Typical values: 0.5-2.0.

**Clipping norm C**: lower C = less noise needed per sample but may bias gradients if many samples have large norms. Typical: auto-clipped to the median gradient norm.

**Batch size**: larger batches amplify privacy (each individual contributes less per batch) and reduce noise-to-signal ratio. DP benefits from large batches.

**Training duration**: fewer epochs = smaller privacy budget consumed. DP-SGD models are often undertrained relative to non-DP counterparts.

In practice, DP-SGD on large datasets (>100K examples) with modern architectures achieves near-equivalent accuracy to non-private training at ε ≈ 3-10. For small datasets or complex tasks, the accuracy gap can be significant.